# Risk and Return Analysis

This notebook uses `data/processed/stock_features.csv` and the reusable functions in `src/financial_metrics.py` to calculate stock-level risk and return metrics, rank symbols, and save presentation-ready charts.


## 1. Setup

Import the analysis libraries, configure project-relative paths, and make sure the reports figures directory exists. The notebook uses **matplotlib only** for visualization.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from financial_metrics import create_stock_metric_summary

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "stock_features.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

DATA_PATH, FIGURES_DIR


## 2. Load Stock Features

Load the feature-engineered stock dataset and sort it by `symbol` and `date` so the return calculations are performed in chronological order for each stock.


In [ ]:
stocks = pd.read_csv(DATA_PATH)
stocks["date"] = pd.to_datetime(stocks["date"], errors="coerce")
stocks = stocks.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"Rows: {stocks.shape[0]:,}")
print(f"Columns: {stocks.shape[1]:,}")
print(f"Symbols: {', '.join(sorted(stocks['symbol'].dropna().unique()))}")
print(f"Date range: {stocks['date'].min().date()} to {stocks['date'].max().date()}")

stocks.head()


## 3. Metric Definitions

- **Cumulative return**: the total compounded return over the full observed period for each stock.
- **Annualized return**: the compounded return converted to a yearly rate, assuming 252 trading days per year.
- **Annualized volatility**: the standard deviation of daily returns scaled to a yearly rate using the square-root-of-time rule.
- **Sharpe ratio**: annualized excess return divided by annualized volatility. This notebook uses the default risk-free rate from `src/financial_metrics.py`.
- **Sortino ratio**: annualized excess return divided by annualized downside deviation, focusing only on negative return volatility.
- **Maximum drawdown**: the worst peak-to-trough percentage decline in the compounded wealth path.
- **VaR 95%**: historical 95% value at risk, reported as the 5th percentile of daily returns.


## 4. Calculate Financial Metrics by Symbol

Use `create_stock_metric_summary` from `src/financial_metrics.py` to calculate one summary row per stock symbol.


In [ ]:
metric_columns = [
    "cumulative_return",
    "annualized_return",
    "annualized_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "var_95",
]

summary = create_stock_metric_summary(stocks)
summary_display = summary[["symbol", *metric_columns]].copy()

percent_columns = [
    "cumulative_return",
    "annualized_return",
    "annualized_volatility",
    "max_drawdown",
    "var_95",
]
summary_display.style.format({column: "{:.2%}" for column in percent_columns}).format({
    "sharpe_ratio": "{:.2f}",
    "sortino_ratio": "{:.2f}",
})


## 5. Rank Stocks by Key Metrics

The rankings below sort stocks by total return, risk-adjusted return, drawdown, and volatility. For maximum drawdown, values closer to zero indicate smaller losses, so the table ranks from least severe to most severe drawdown.


In [ ]:
def display_ranking(metric, ascending=False, title=None):
    """Display a ranked table for a selected metric."""
    ranking = (
        summary[["symbol", metric]]
        .sort_values(metric, ascending=ascending)
        .reset_index(drop=True)
    )
    ranking.insert(0, "rank", ranking.index + 1)
    if title:
        print(title)
    display(ranking)
    return ranking

cumulative_return_ranking = display_ranking(
    "cumulative_return",
    ascending=False,
    title="Ranking by cumulative return (highest to lowest)",
)

sharpe_ratio_ranking = display_ranking(
    "sharpe_ratio",
    ascending=False,
    title="Ranking by Sharpe ratio (highest to lowest)",
)

max_drawdown_ranking = display_ranking(
    "max_drawdown",
    ascending=False,
    title="Ranking by maximum drawdown (least severe to most severe)",
)

annualized_volatility_ranking = display_ranking(
    "annualized_volatility",
    ascending=True,
    title="Ranking by annualized volatility (lowest to highest)",
)


## 6. Chart Helper

Define a reusable single-chart helper for ranked bar charts. Each chart is created as its own matplotlib figure and saved to `reports/figures`.


In [ ]:
def save_bar_chart(df, metric, title, ylabel, filename, ascending=False):
    """Create and save a single ranked bar chart for one metric."""
    plot_data = df[["symbol", metric]].sort_values(metric, ascending=ascending)

    fig = plt.figure()
    plt.bar(plot_data["symbol"], plot_data[metric])
    plt.title(title)
    plt.xlabel("Symbol")
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved chart to {output_path}")


## 7. Cumulative Return Bar Chart

This chart compares each stock's total compounded return over the available history.


In [ ]:
save_bar_chart(
    summary,
    metric="cumulative_return",
    title="Cumulative Return by Symbol",
    ylabel="Cumulative Return",
    filename="risk_return_cumulative_return_bar.png",
    ascending=False,
)


## 8. Sharpe Ratio Bar Chart

This chart compares risk-adjusted performance using return per unit of total volatility.


In [ ]:
save_bar_chart(
    summary,
    metric="sharpe_ratio",
    title="Sharpe Ratio by Symbol",
    ylabel="Sharpe Ratio",
    filename="risk_return_sharpe_ratio_bar.png",
    ascending=False,
)


## 9. Maximum Drawdown Bar Chart

This chart shows each stock's largest historical peak-to-trough decline during the observed period. More negative values indicate deeper drawdowns.


In [ ]:
save_bar_chart(
    summary,
    metric="max_drawdown",
    title="Maximum Drawdown by Symbol",
    ylabel="Maximum Drawdown",
    filename="risk_return_max_drawdown_bar.png",
    ascending=False,
)


## 10. Annualized Volatility Bar Chart

This chart compares annualized return variability. Lower volatility may indicate smoother returns, while higher volatility indicates a wider range of outcomes.


In [ ]:
save_bar_chart(
    summary,
    metric="annualized_volatility",
    title="Annualized Volatility by Symbol",
    ylabel="Annualized Volatility",
    filename="risk_return_annualized_volatility_bar.png",
    ascending=True,
)


## 11. Risk-Return Scatterplot

The scatterplot places annualized volatility on the x-axis and annualized return on the y-axis. Stocks farther up have higher annualized returns, while stocks farther right have higher annualized volatility.


In [ ]:
plot_data = summary.dropna(subset=["annualized_volatility", "annualized_return"])

fig = plt.figure()
plt.scatter(
    plot_data["annualized_volatility"],
    plot_data["annualized_return"],
    s=100,
)

for _, row in plot_data.iterrows():
    plt.annotate(
        row["symbol"],
        (row["annualized_volatility"], row["annualized_return"]),
        textcoords="offset points",
        xytext=(6, 6),
    )

plt.title("Risk-Return Scatterplot by Symbol")
plt.xlabel("Annualized Volatility")
plt.ylabel("Annualized Return")
plt.tight_layout()

output_path = FIGURES_DIR / "risk_return_scatterplot.png"
fig.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved chart to {output_path}")


## 12. Risk and Return Takeaways

Use this section after running the notebook to document actual observations from the generated tables and charts.

- **Highest cumulative return:** _Add observed symbol and interpretation here._
- **Best Sharpe ratio:** _Add observed symbol and interpretation here._
- **Smallest maximum drawdown:** _Add observed symbol and interpretation here._
- **Lowest annualized volatility:** _Add observed symbol and interpretation here._
- **Overall risk-return profile:** _Add portfolio or stock-selection takeaway here._
- **Follow-up questions:** _Add any follow-up analysis ideas here._
